# pricing: E5-base regression → LightGBM
Choose **Runtime → Change runtime type → GPU** before starting. This notebook fine-tunes all transformer weights on descriptions and log prices. Three-fold out-of-fold predictions protect the stacking features from target leakage.

Data: historical US Craigslist asking prices from April–May 2021, not verified sale prices. A fixed 10% sample contains 42,688 of 426,880 raw rows. Filtering and deduplication leave 31,001 eligible/deduplicated rows: 21,700 training / 4,650 validation / 4,651 test. The text stage trains on all training rows; Isolation Forest filtering applies only to LightGBM. Training downloads a large checkpoint and the source dataset. It may take substantial GPU time. Keep the output folder to resume completed folds.


## Predeclared sweep
6 Huber configurations: Isolation Forest on the single out-of-fold predicted_text_price scalar, contamination="auto", 300 trees × dollar/log1p targets × Huber thresholds 0.5, 0.9 and 1.5. The fitted detector determines the removal count; no fraction is forced. Targets are standardized on retained training rows so thresholds are meaningful. Validation selects the MAE and RMSE champions before test scoring. The GUI deploys the validation-MAE champion. This large full-transformer run requires substantial GPU time; it is not a quick CPU experiment.


In [ ]:
!git clone https://github.com/botsandbolts/Pricing.git /content/pricing
%cd /content/pricing
!pip -q install -r requirements-training.txt
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
source=Path('/content/vehicles.zip')
if not source.exists():
    urlretrieve('https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1', source)
!python -m scripts.prepare_stack /content/vehicles.zip --fraction .1 --output training-cache/ten-percent.jsonl


In [ ]:
!python -m pricing.train --data training-cache/ten-percent.jsonl --output artifacts/e5-base-isolation-text-10 --device cuda --folds 3 --epochs 2 --batch-size 1 --accumulation 8 --text-only
!python -m pricing.experiments --data training-cache/ten-percent.jsonl --output artifacts/e5-base-isolation-text-10 --trees 1000 --losses huber --huber-deltas .5 .9 1.5
from pathlib import Path
assert Path('artifacts/e5-base-isolation-text-10/manifest.json').exists(), 'Training did not finish; inspect the log.'


In [ ]:
import json
e=json.loads(Path('artifacts/e5-base-isolation-text-10/evaluation.json').read_text())
print(json.dumps(e,indent=2))


## Save the model, then run the GUI
Download the completed bundle and unpack it in the repository root. Start the app locally with `python -m streamlit run app.py`. No public inference endpoint or tunnel is created. The bundle includes the fine-tuned transformer (~1 GB), regression head, LightGBM, schemas and evaluation. Colab storage is temporary; download the bundle before leaving.


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/pricing-e5-base-lightgbm','zip','artifacts')
files.download(archive)
